# C03. Cross-entropy 손실 함수

> 📌 **이 모듈에서 배울 것**
> 
> 분류 문제에서 MSE 대신 사용하는 손실 함수.  
> **확률**과 **라벨**의 차이를 측정.

## 사전 지식

- C01 (분류 vs 회귀), C02 (시그모이드) 완료

---


## 1. 왜 MSE가 아닌가?

C01에서 봤듯이 분류에 MSE를 쓰면 어색해요. **확률을 평가**하는 데 더 적합한 손실 함수가 필요.

핵심 아이디어:
- 예측 확률이 라벨과 가까우면 → 손실 작음
- 예측 확률이 라벨과 멀면 → 손실 큼
- 특히 **"틀린 걸 자신 있게 말하면" 큰 페널티**


## 2. Cross-entropy 식

이진 분류 (Adelie=0, Gentoo=1)에서:

$$E = -[y \log(\hat{y}) + (1-y) \log(1-\hat{y})]$$

- $y$: 실제 라벨 (0 또는 1)
- $\hat{y}$: 모델 예측 확률 (시그모이드 출력, 0~1)

복잡해 보이지만 직관 단순해요. 실제 라벨이 0이냐 1이냐에 따라:
- $y=1$이면: $E = -\log(\hat{y})$ ← 예측이 1에 가까울수록 손실 작음
- $y=0$이면: $E = -\log(1-\hat{y})$ ← 예측이 0에 가까울수록 손실 작음


## 3. $-\log p$ 그래프 — 핵심 직관


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

p = np.linspace(0.001, 1, 200)
neg_log_p = -np.log(p)

plt.figure(figsize=(8, 4))
plt.plot(p, neg_log_p, linewidth=2)
plt.xlabel('p (predicted probability)')
plt.ylabel('-log(p)')
plt.title('Loss when true label is 1')
plt.grid(alpha=0.3)
plt.axvline(1.0, color='green', linestyle='--', alpha=0.5, label='p=1 (correct)')
plt.axvline(0.1, color='red', linestyle='--', alpha=0.5, label='p=0.1 (very wrong)')
plt.legend()
plt.show()


**직관**:
- `p = 1.0` (정확) → 손실 ≈ 0 ✅
- `p = 0.5` (확신 없음) → 손실 ≈ 0.69
- `p = 0.1` (틀림) → 손실 ≈ 2.3
- `p → 0` (완전히 틀림) → 손실 ≈ ∞

**핵심**: "맞으면 0, 틀리면 무한대 벌점". 자신 있게 틀린 걸 강하게 처벌.


## 4. MSE vs Cross-entropy 비교


In [ ]:
y_true = 1.0  # 실제 라벨 = 1

p_range = np.linspace(0.001, 0.999, 200)
mse = (y_true - p_range) ** 2
ce = -(y_true * np.log(p_range) + (1 - y_true) * np.log(1 - p_range))

plt.figure(figsize=(8, 5))
plt.plot(p_range, mse, label='MSE: (1 - p)²', linewidth=2)
plt.plot(p_range, ce, label='Cross-entropy: -log(p)', linewidth=2)
plt.xlabel('p (predicted probability when true label = 1)')
plt.ylabel('Loss')
plt.title('MSE vs Cross-entropy when y=1')
plt.legend()
plt.grid(alpha=0.3)
plt.ylim(0, 5)
plt.show()


**차이 보이세요?**
- p가 1에 가까우면 (정확): 둘 다 손실 작음
- p가 0에 가까우면 (틀림): **CE가 훨씬 빠르게 증가** → 강한 페널티

이게 분류에서 CE가 MSE보다 학습을 잘 시키는 이유.


## 5. PyTorch에서 Cross-entropy


In [ ]:
import torch

# 예시 데이터
y_true = torch.tensor([1.0, 0.0, 1.0, 0.0])  # 진짜 라벨
y_pred = torch.tensor([0.9, 0.1, 0.6, 0.4])  # 시그모이드 출력 (확률)

# 손실 직접 계산
loss = -(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred)).mean()
print(f"손실: {loss.item():.4f}")


In [ ]:
# PyTorch 내장 함수 — F.binary_cross_entropy
import torch.nn.functional as F

loss_pt = F.binary_cross_entropy(y_pred, y_true)
print(f"PyTorch 내장: {loss_pt.item():.4f}")  # 같은 결과


**더 안전한 방법**: `F.binary_cross_entropy_with_logits` — 시그모이드 + CE 한 번에. 수치 안정성 더 좋음.


In [ ]:
# 시그모이드 적용 전의 raw 출력 (logits)
y_logit = torch.tensor([2.2, -2.2, 0.4, -0.4])

# 자동으로 시그모이드 + CE
loss_safe = F.binary_cross_entropy_with_logits(y_logit, y_true)
print(f"with_logits: {loss_safe.item():.4f}")

# 같은 결과인지 확인
loss_manual = -(y_true * torch.log(torch.sigmoid(y_logit)) +
                (1 - y_true) * torch.log(1 - torch.sigmoid(y_logit))).mean()
print(f"수동: {loss_manual.item():.4f}")


## 6. 핵심 정리 — 회귀와 분류의 코드 차이

다음 모듈(C04)에서 적용할 핵심:

| | 회귀 | 분류 |
|---|---|---|
| 출력 | `y_hat = X @ w + b` | `y_hat = torch.sigmoid(X @ w + b)` |
| 손실 | `((y - y_hat)**2).mean()` | `-(y * log(y_hat) + (1-y) * log(1-y_hat)).mean()` |

**코드 차이는 두 줄!** C04에서 실제로 만져봅시다.


## 7. ⚠️ 함정 / 주의사항

### 7.1 log(0)은 -∞
예측이 정확히 0 또는 1이면 log(0)이 발생. 수치 폭발.  
**해결**: `F.binary_cross_entropy_with_logits` (안전한 구현 포함)

### 7.2 라벨 타입
`y_true`가 float이어야 함 (long/int면 에러).
```python
y_true = y_true.float()
```

### 7.3 다중 분류는 다름
3개 이상 클래스는 **categorical cross-entropy** + **softmax** 사용.  
PyTorch: `F.cross_entropy()` (소프트맥스 포함)


## 8. 📚 더 알아보기 (정공법 트랙 — 다음 학기 떡밥)

이 모듈은 "왜 cross-entropy인가"를 직관으로만 받아들였어요.  
진짜 깊이 들어가려면:

- **최대우도추정 (MLE)** 관점: cross-entropy = -log-likelihood
- **정보이론** 관점: KL divergence, 정보량 $\log(1/p)$
- **편미분 유도**: 결과가 회귀와 똑같이 $\sum(\hat{y}-y)x$로 떨어짐 (1·2차시 미분 정신 그대로)

다음 학기 또는 자율 학습 주제!
